Introduction

Retrieval-Augmented Generation (RAG) is a powerful approach that combines the strengths of retrieval-based methods and generative models to enhance the quality and relevance of generated content. RAG works by first retrieving relevant information from a large corpus of data and then using that information to generate more accurate and contextually appropriate responses.

1. Langchain - a framework for building applications with language models, provides tools and components to implement RAG effectively. It allows developers to create applications that can retrieve information from various sources and generate responses based on that information.
2. Chromadb - an open-source vector database that can be used to store and retrieve high-dimensional data, such as embeddings generated by language models. Chromadb can be integrated with RAG systems to efficiently manage and query large datasets, enabling faster retrieval of relevant information for generation tasks.
3. OpenAI - for embeddings and language models, offers powerful APIs that can be utilized in RAG systems. OpenAI's models can generate high-quality embeddings for retrieval tasks and produce coherent and contextually relevant text for generation tasks.

In [23]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [24]:
# Langchain imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

# vector store imports
from langchain_community.vectorstores import Chroma

# utility imports
from typing import List
import numpy as np

In [25]:
# RAG Architecture

# 1.Document Loading: Load documents from various sources (e.g., PDFs, web pages, databases) and convert them into a format suitable for processing.
# 2.Document Splitting: Split large documents into smaller chunks to ensure they fit within the context window of language models and to improve retrieval efficiency.
# 3.Embedding Generation: Convert text chunks into vector representations (embeddings) using models like OpenAI's embedding models.
# 4.Vector Store Creation: Store the generated embeddings in a vector database (e.g.,
# 5.Query Processing: When a user query is received, convert the query into an embedding and use it to retrieve relevant document chunks from the vector store.
# 6.Similarity Search: Perform a similarity search in the vector store to find the most relevant document chunks based on the query embedding.
# 7.Context Augmentation: Combine the retrieved document chunks with the original user query to create an augmented input for the language model.
# 8.Response Generation: Use the augmented input to generate a response using a language model, providing more accurate and contextually relevant answers.


1. Sample Data

In [26]:
# create sample documents
sample_docs = [
    """
    Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that focuses on the development of algorithms that can learn from and make predictions on data. It involves training models on large datasets to recognize patterns and make informed decisions without being explicitly programmed for specific tasks.
    The process of machine learning typically involves several key steps:
    1. Data Collection: Gathering relevant data that will be used to train the machine learning
    model. This data can come from various sources, such as databases, web scraping, or sensors.
    2. Data Preprocessing: Cleaning and preparing the data for training. This may involve
    handling missing values, encoding categorical variables, and normalizing numerical features.
    3. Model Selection: Choosing an appropriate machine learning algorithm based on the
    problem at hand (e.g., classification, regression, clustering).
    4. Training: Feeding the preprocessed data into the selected model and allowing it to learn from the data by adjusting its parameters to minimize the error between predictions and actual outcomes.
""",
    """
    Deep Learning Overview

    Deep learning is a subset of machine learning that focuses on neural networks with many layers (deep neural networks). It has been particularly successful in tasks such as image recognition, natural language processing, and speech recognition. Deep learning models are capable of automatically learning hierarchical representations of data, which allows them to capture complex patterns and relationships.
    The architecture of deep learning models typically consists of an input layer, multiple hidden layers, and an output layer. Each layer contains a set of neurons that process the input data and pass the information to the next layer. The training process involves adjusting the weights of the connections between neurons to minimize the error in predictions.       
    The success of deep learning can be attributed to the availability of large datasets, advancements in computational power (especially GPUs), and the development of more sophisticated algorithms and architectures (e.g., convolutional neural networks, recurrent neural networks, transformers).
    Its applications span across various domains, including healthcare (e.g., medical image analysis), finance (e.g., fraud detection), and autonomous systems (e.g., self-driving cars).
""",
"""
    Natural Language Processing (NLP) Basics
    
    Natural language processing (NLP) is a field of artificial intelligence that focuses on the interaction between computers and human language. It involves the development of algorithms and models that enable computers to understand, interpret, and generate human language in a meaningful way.
    NLP encompasses a wide range of tasks, including:
    1. Text Classification: Assigning predefined categories to text (e.g., spam detection, sentiment analysis).
    2. Named Entity Recognition (NER): Identifying and classifying entities in text (e.g., people, organizations, locations).
    3. Machine Translation: Automatically translating text from one language to another.        
    4. Text Summarization: Generating concise summaries of longer texts while preserving key information.
"""
]
sample_docs

['\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that focuses on the development of algorithms that can learn from and make predictions on data. It involves training models on large datasets to recognize patterns and make informed decisions without being explicitly programmed for specific tasks.\n    The process of machine learning typically involves several key steps:\n    1. Data Collection: Gathering relevant data that will be used to train the machine learning\n    model. This data can come from various sources, such as databases, web scraping, or sensors.\n    2. Data Preprocessing: Cleaning and preparing the data for training. This may involve\n    handling missing values, encoding categorical variables, and normalizing numerical features.\n    3. Model Selection: Choosing an appropriate machine learning algorithm based on the\n    problem at hand (e.g., classification, regression, clustering).\n    4. Training: Feeding the prep

In [27]:
# # Save Sample Documents to Text Files
# import tempfile
# tempdir = tempfile.mkdtemp()

# for i, doc in enumerate(sample_docs):
#     with open(os.path.join(tempdir, f"doc_{i}.txt"), "w") as f:
#         f.write(doc)
# print(f"Sample documents saved to: {tempdir}")

In [28]:
# Save sample documents to text files
for i, doc in enumerate(sample_docs):
    with open(f"doc_{i}.txt", "w") as f:
        f.write(doc)
print("Sample documents saved to current directory as doc_0.txt, doc_1.txt, and doc_2.txt")

Sample documents saved to current directory as doc_0.txt, doc_1.txt, and doc_2.txt


Document Loading

In [29]:
from langchain_core.documents import Document
from langchain_community.document_loaders import DirectoryLoader

# Load documents from directory
loader = DirectoryLoader(
    "data",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)
documents = loader.load()

print(f"Loaded {len(documents)} documents:")
print(f"\nFirst document content:\n{documents[0].page_content[:500]}")  # Print first 500 characters of the first document

Loaded 3 documents:

First document content:

    Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that focuses on the development of algorithms that can learn from and make predictions on data. It involves training models on large datasets to recognize patterns and make informed decisions without being explicitly programmed for specific tasks.
    The process of machine learning typically involves several key steps:
    1. Data Collection: Gathering relevant data that will be used to train the mac


Document SPlitting

In [30]:
# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50, length_function=len, separators=[" "])
chunks = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(chunks)}")
print(f"\nFirst chunk content:\n{chunks[0].page_content[:500]}")  # Print first 500 characters of the first chunk
print(f"\nMetadata of first chunk:\n{chunks[0].metadata}")

Total chunks created: 8

First chunk content:
Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that focuses on the development of algorithms that can learn from and make predictions on data. It involves training models on large datasets to recognize patterns and make informed decisions without being explicitly programmed for specific tasks.
    The process of machine learning typically involves several key steps:
    1. Data Collection: Gathering relevant data that will be used to train the

Metadata of first chunk:
{'source': 'data\\doc_0.txt'}


In [31]:
chunks

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that focuses on the development of algorithms that can learn from and make predictions on data. It involves training models on large datasets to recognize patterns and make informed decisions without being explicitly programmed for specific tasks.\n    The process of machine learning typically involves several key steps:\n    1. Data Collection: Gathering relevant data that will be used to train the'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='relevant data that will be used to train the machine learning\n    model. This data can come from various sources, such as databases, web scraping, or sensors.\n    2. Data Preprocessing: Cleaning and preparing the data for training. This may involve\n    handling missing values, encoding categorical variables, and normalizing numerical features.\n    3. Model Selection:

Embedding Models

In [32]:
from langchain_huggingface import HuggingFaceEmbeddings


sample_text = """MAchine learning is a subset of artificial intelligence that focuses on the development of algorithms that can learn from and make predictions on data. It involves training models on large datasets to recognize patterns and make informed decisions without being explicitly programmed for specific tasks."""
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# convert sample text to embedding vector
sample_embedding = embeddings.embed_query(sample_text)
sample_embedding[:10]  # Print first 10 dimensions of the embedding vector

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4885.65it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[-0.018290797248482704,
 0.0009892560774460435,
 0.03552750498056412,
 0.010352895595133305,
 0.046883828938007355,
 -0.026274142786860466,
 -0.018024705350399017,
 -0.04563027620315552,
 -0.08344074338674545,
 -0.004454660229384899]

Initialize a chromdb vector store and store chunks in visual representation

In [33]:
# Create a chromadb vector store
persist_directory = "./chromadb"

# Initialize Chromadb with openai embeddings
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=HuggingFaceEmbeddings(),
    persist_directory=persist_directory,
    collection_name="rag_collection"
)
print(f"Vector Stored created with {vectorstore._collection.count()} vectors")
print(f"Persisted to: {persist_directory}")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5670.25it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector Stored created with 24 vectors
Persisted to: ./chromadb


Test Similarity Search

In [34]:
query = "What are types of machine learning?"

similar_DOCS = vectorstore.similarity_search(query, k=3)
similar_DOCS

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that focuses on the development of algorithms that can learn from and make predictions on data. It involves training models on large datasets to recognize patterns and make informed decisions without being explicitly programmed for specific tasks.\n    The process of machine learning typically involves several key steps:\n    1. Data Collection: Gathering relevant data that will be used to train the'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that focuses on the development of algorithms that can learn from and make predictions on data. It involves training models on large datasets to recognize patterns and make informed decisions without being explicitly programmed for specific tasks.\n    The process of machine lear

In [41]:
query = "What is nlp?"

similar_DOCS = vectorstore.similarity_search(query)
similar_DOCS

[Document(metadata={'source': 'data\\doc_2.txt'}, page_content='Natural Language Processing (NLP) Basics\n\n    Natural language processing (NLP) is a field of artificial intelligence that focuses on the interaction between computers and human language. It involves the development of algorithms and models that enable computers to understand, interpret, and generate human language in a meaningful way.\n    NLP encompasses a wide range of tasks, including:\n    1. Text Classification: Assigning predefined categories to text (e.g., spam detection, sentiment'),
 Document(metadata={'source': 'data\\doc_2.txt'}, page_content='Natural Language Processing (NLP) Basics\n\n    Natural language processing (NLP) is a field of artificial intelligence that focuses on the interaction between computers and human language. It involves the development of algorithms and models that enable computers to understand, interpret, and generate human language in a meaningful way.\n    NLP encompasses a wide rang

Advanced similarity search with scores

In [43]:
result_scores = vectorstore.similarity_search_with_score(query, k=3)
result_scores

[(Document(metadata={'source': 'data\\doc_2.txt'}, page_content='Natural Language Processing (NLP) Basics\n\n    Natural language processing (NLP) is a field of artificial intelligence that focuses on the interaction between computers and human language. It involves the development of algorithms and models that enable computers to understand, interpret, and generate human language in a meaningful way.\n    NLP encompasses a wide range of tasks, including:\n    1. Text Classification: Assigning predefined categories to text (e.g., spam detection, sentiment'),
  0.5642641186714172),
 (Document(metadata={'source': 'data\\doc_2.txt'}, page_content='Natural Language Processing (NLP) Basics\n\n    Natural language processing (NLP) is a field of artificial intelligence that focuses on the interaction between computers and human language. It involves the development of algorithms and models that enable computers to understand, interpret, and generate human language in a meaningful way.\n    NL

Initiliaze LLM, RAG Chain, Prompt template, Query the RAG system

In [44]:
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_length=200
)

llm = HuggingFacePipeline(pipeline=pipe)

print(llm.invoke("Explain what RAG is in simple terms"))

e:\Ultimate RAG Bootcamp Using Langchain, LangGraph & Langsmith\RAGUDEMY\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub\models--google--flan-t5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 282/282 [00:00<00:00, 3705.83it/s]
The 

Explain what RAG is in simple termsimarily a part of a particular group of people who have a specialized medical condition that is determined primarily by the medical expert.


In [50]:
llm

HuggingFacePipeline(pipeline=TextGenerationPipeline: {'model': 'T5ForConditionalGeneration', 'dtype': 'float32', 'device': 'cpu', 'input_modalities': 'text', 'output_modalities': ('text',)})

In [61]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

pipe = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200
)

print(pipe("Explain RAG in simple terms"))

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 4834.73it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


KeyError: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"